# Average a metric across superclass CSVs

Reads every `*.csv` in `analysis/data/resnet50`, extracts one column, and averages it per `(Tau, m)` combination.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data/resnet50")
METRICS = [  # <- change to the columns you want
    "val_test_genus_knn_top1",
    "val_test_genus_knn_top5",
    "val_cophenetic_cpcc",
]

files = sorted(DATA_DIR.glob("*.csv"))
print(f"{len(files)} files:", [f.stem for f in files])


In [ ]:
# Available metric columns (from the first file)
list(pd.read_csv(files[0]).columns)

In [ ]:
TAU_ZERO_INF_LABEL = "infinite"  # tau=0 and tau=infinity denote the same setting
INF_ALIASES = {"inf", "+inf", "-inf", "infty", "infinity", "infinite", "∞"}


def normalize_key(value) -> str:
    """Collapse equivalent numeric labels (1 vs 1.0 vs 1.00) to one string; blanks -> '-'."""
    if pd.isna(value) or str(value).strip() == "":
        return "-"
    try:
        return f"{float(value):g}"
    except ValueError:
        return str(value).strip()


def normalize_tau(value) -> str:
    text = str(value).strip().lower()
    if text in INF_ALIASES:
        return TAU_ZERO_INF_LABEL
    key = normalize_key(value)
    if key in {"0", "-0", "inf", "-inf"}:
        return TAU_ZERO_INF_LABEL
    return key


def load(path: Path, metrics: list[str]) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={"Tau": str, "m": str})
    missing = [m for m in metrics if m not in df.columns]
    if missing:
        raise KeyError(f"{missing} not in {path.name}")
    df = df[["Tau", "m", *metrics]].copy()
    df["Tau"] = df["Tau"].map(normalize_tau)
    df["m"] = df["m"].map(normalize_key)
    for metric in metrics:
        df[metric] = pd.to_numeric(df[metric], errors="coerce")
    df["source"] = path.stem.split("-")[0]
    return df


long = pd.concat([load(f, METRICS) for f in files], ignore_index=True)
long.head()


In [ ]:
def sort_key(value: str) -> float:
    try:
        return float(value)
    except ValueError:
        return float("-inf")


# One per-file table per metric, keyed by metric name
tables = {}
for metric in METRICS:
    t = long.pivot_table(index=["Tau", "m"], columns="source", values=metric, aggfunc="mean")
    t["mean"] = t.mean(axis=1)
    t["n_files"] = long.groupby(["Tau", "m"])[metric].count()
    tables[metric] = t.sort_index(key=lambda idx: idx.map(sort_key), ascending=False)

tables[METRICS[0]].round(4)


In [ ]:
# Compact table: Tau and m as separate columns, one column per metric (averaged over files)
summary = long.groupby(["Tau", "m"])[METRICS].mean()
summary = summary.sort_index(key=lambda idx: idx.map(sort_key), ascending=False).reset_index()
summary.round(4)


In [ ]:
out = DATA_DIR.parent / "avg_metrics_by_tau_m.csv"
summary.round(6).to_csv(out, index=False)
print("wrote", out)

for metric, t in tables.items():
    path = DATA_DIR.parent / f"avg_{metric}_by_tau_m.csv"
    t.round(6).to_csv(path)
    print("wrote", path)
